# Multi-Model Content Pipeline

> A single Ray Serve application with **4 independently scaling deployments** on Anyscale.

[![Python 3.11+](https://img.shields.io/badge/python-3.11%2B-blue.svg)](https://www.python.org/downloads/)
[![Ray Serve](https://img.shields.io/badge/ray--serve-2.53-blue.svg)](https://docs.ray.io/en/latest/serve/)
[![License: MIT](https://img.shields.io/badge/license-MIT-green.svg)](LICENSE)

Each deployment (content filter, sentiment classifier, summarizer, orchestrator) declares its own CPU/GPU resources and autoscaling rules. The Ray scheduler places replicas on matching nodes — CPU traffic spikes don't waste GPU resources.

**Share & run:** Clone the repo and open **[`demo.ipynb`](demo.ipynb)** for a step-by-step deploy and test flow, or run the app locally with the commands below.

## Table of Contents

- [Run the demo notebook](#run-the-demo-notebook)
- [Architecture](#architecture)
  - [Infrastructure Topology](#1-infrastructure-topology)
  - [Deployment Mapping](#2-deployment--infrastructure-mapping)
  - [Request Flow](#3-request-flow)
  - [Deployment Pipeline](#4-deployment-pipeline-code--production)
  - [Autoscaling Behavior](#5-independent-autoscaling)
- [Quick Start](#quick-start)
- [Deployments](#deployments)
- [Models](#models)
- [Instance Types](#instance-types)
- [Design Reasoning](#design-reasoning)
- [Customization](#customization)
- [Files](#files)
- [References](#references)

## Architecture

> GitHub renders the Mermaid blocks below as diagrams. Static PNGs are also available in [`diagrams/`](diagrams/) for use outside GitHub.

### 1. Infrastructure Topology

The Anyscale cluster uses three node types. The head node runs the control plane; GPU and CPU worker pools scale independently based on pending Ray resource requests.

![Infrastructure Topology](diagrams/diagram_1_infrastructure_topology.png)

<details>
<summary>Mermaid source</summary>

```mermaid
graph TB
    subgraph cluster["Anyscale Cluster"]

        subgraph head["Head Node · 16CPU-64GB"]
            controller["Ray Serve Controller"]
            gcs["GCS · Dashboard · Autoscaler"]
        end

        subgraph gpu_pool["GPU Workers · A10-24G<br/>(24 GB VRAM) · 36 vCPU"]
            gw1["Worker 1"]
            gw2["Worker 2"]
            gw3["Worker 3"]
            gw4["Worker 4"]
        end

        subgraph cpu_pool["CPU Workers<br/>16CPU · 64GB"]
            cw1["Worker 1"]
            cw2["Worker 2"]
            cw_n["Worker ···4"]
        end

    end

    controller -- "schedule GPU replicas" --> gpu_pool
    controller -- "schedule CPU replicas" --> cpu_pool

    style head fill:#e8f0fe,stroke:#4285f4
    style gpu_pool fill:#fef3e0,stroke:#f9a825
    style cpu_pool fill:#e8f5e9,stroke:#43a047
    style cluster fill:#f8f9fa,stroke:#dadce0
```

</details>

### 2. Deployment → Infrastructure Mapping

Each deployment declares CPU/GPU resources. The scheduler places replicas on matching nodes. Fractional GPU (0.25) packs up to 4 replicas per model onto one physical A10.

![Deployment → Infrastructure Mapping](diagrams/diagram_2_deployment_mapping.png)

<details>
<summary>Mermaid source</summary>

```mermaid
graph LR
    subgraph serve_app["serve_app:app"]
        CP["ContentPipeline<br/>(Ingress)<br/>1 CPU · 1–5 replicas"]
        CF["ContentFilter<br/>1 CPU · 1–8 replicas"]
        SC["SentimentClassifier<br/>0.25 GPU + 1 CPU<br/>1–4 replicas"]
        SUM["Summarizer<br/>0.25 GPU + 1 CPU<br/>1–4 replicas"]
    end

    subgraph gpu_nodes["A10-24G GPU Node"]
        g1["Up to 8 replicas<br/>(Sentiment + Summarizer)<br/>sharing 0.25 GPU each"]
    end

    subgraph cpu_nodes["CPU Nodes · 16CPU-64GB"]
        c1["Nodes 1–4"]
    end

    CP -. "scheduled on" .-> cpu_nodes
    CF -. "scheduled on" .-> cpu_nodes
    SC -. "scheduled on" .-> gpu_nodes
    SUM -. "scheduled on" .-> gpu_nodes

    style serve_app fill:#f3e5f5,stroke:#8e24aa
    style gpu_nodes fill:#fef3e0,stroke:#f9a825
    style cpu_nodes fill:#e8f5e9,stroke:#43a047
```

</details>

### 3. Request Flow

The orchestrator runs a cheap CPU filter first (early rejection saves GPU cost), then fans out to both GPU models in parallel. Total latency ≈ `max(sentiment, summary)` ≈ 200 ms – 1 s.

![Request Flow](diagrams/diagram_3_request_flow.png)

<details>
<summary>Mermaid source</summary>

```mermaid
sequenceDiagram
    autonumber
    participant Client
    participant CP as ContentPipeline<br/>(CPU)
    participant CF as ContentFilter<br/>(CPU)
    participant SC as SentimentClassifier<br/>(0.25 GPU)
    participant SUM as Summarizer<br/>(0.25 GPU)

    Client->>CP: POST /analyze {text}

    Note right of CF: ~1 ms · CPU only
    CP->>CF: filter.remote(text)
    CF-->>CP: {passed, cleaned_text, pii}

    alt Content rejected
        CP-->>Client: {status: rejected} — no GPU cost
    else Content passed
        par asyncio.gather (parallel)
            Note right of SC: ~10 ms
            CP->>SC: classify.remote(cleaned_text)
            Note right of SUM: ~200 ms
            CP->>SUM: summarize.remote(cleaned_text)
        end
        SC-->>CP: {label, score}
        SUM-->>CP: {summary, tokens}
        CP-->>Client: {status: processed, filter, sentiment, summary}
    end
```

</details>

### 4. Deployment Pipeline (Code → Production)

![Deployment Pipeline](diagrams/diagram_4_deployment_pipeline.png)

<details>
<summary>Mermaid source</summary>

```mermaid
flowchart LR
    subgraph dev["Developer"]
        code["serve_app.py<br/>+ requirements.txt"]
        config["service.yaml"]
    end

    subgraph cli["Anyscale CLI"]
        deploy["anyscale service deploy<br/>-f service.yaml"]
    end

    subgraph anyscale["Anyscale Platform"]
        build["Build container image<br/>(ray-llm:2.56.0-py312-cu130<br/>+ pip requirements)"]
        provision["Provision cluster<br/>(head: 16CPU-64GB<br/>+ auto-select workers)"]
        start["Start Ray Serve<br/>application with<br/>4 deployments"]
        live["Service live<br/>HTTPS endpoint<br/>+ autoscaling"]
    end

    code --> deploy
    config --> deploy
    deploy --> build --> provision --> start --> live

    style dev fill:#e8f0fe,stroke:#4285f4
    style cli fill:#fff3e0,stroke:#ff8f00
    style anyscale fill:#e8f5e9,stroke:#43a047
```

</details>

### 5. Independent Autoscaling

Each deployment tier scales to its own bottleneck. CPU traffic spikes do not waste GPU resources.

![Independent Autoscaling](diagrams/diagram_5_autoscaling.png)

<details>
<summary>Mermaid source</summary>

```mermaid
graph TB
    subgraph scaling["Independent Autoscaling per Deployment"]

        subgraph cpu_tier["CPU Tier"]
            CF_S["ContentFilter<br/>CPU only · ~1 ms<br/>1 → 8 replicas"]
            CP_S["ContentPipeline<br/>CPU only · orchestrator<br/>1 → 5 replicas"]
        end

        subgraph gpu_tier["GPU Tier · shared A10-24G"]
            SC_S["SentimentClassifier<br/>0.25 GPU · ~10 ms<br/>1 → 4 replicas"]
            SUM_S["Summarizer<br/>0.25 GPU · ~200 ms<br/>1 → 4 replicas"]
        end

    end

    traffic["Incoming Traffic"] --> CF_S
    CF_S -->|"passed requests"| CP_S
    CP_S -->|"parallel fan-out"| SC_S
    CP_S -->|"parallel fan-out"| SUM_S

    style cpu_tier fill:#e8f5e9,stroke:#43a047
    style gpu_tier fill:#fef3e0,stroke:#f9a825
    style scaling fill:#f8f9fa,stroke:#dadce0
```

</details>

> **Key insight**: Under high traffic, ContentFilter might run 8 replicas while Summarizer stays at 1–4. You don't pay for GPU replicas just because CPU traffic spiked.

## Run the demo notebook

The **[`demo.ipynb`](demo.ipynb)** notebook walks through:

1. Inspecting the service config and application code  
2. Deploying to Anyscale (`anyscale service deploy -f service.yaml`)  
3. Waiting for the service and calling `/health` and `/analyze`  
4. Testing content filtering, sentiment, and summarization  
5. Optional: throughput test and controller logs  

**Where to run it:** Anyscale workspace (recommended), or locally with `anyscale` CLI and credentials configured. Open the notebook in Jupyter or VS Code and run cells in order.

## Quick Start

### Prerequisites

- Python 3.11+
- GPU with CUDA support (A10 or similar)
- No HuggingFace token needed (ungated models)

### Clone and open the notebook

```bash
git clone https://github.com/YOUR_ORG/multi-model-content-pipeline.git
cd multi-model-content-pipeline
# Open demo.ipynb in Jupyter or VS Code and run the cells
```

Replace `YOUR_ORG/multi-model-content-pipeline` with your repo URL after pushing to GitHub.

### Set up an Anyscale workspace (CPU-only)

Create a **CPU-only** workspace to load the code, install dependencies, and iterate — no GPU is assigned (the GPU replicas come into play when you deploy the service). These commands only **create** the workspace; you start it manually afterwards.

```bash
# 1. Register the CPU-only compute config
anyscale compute-config create -n workspace-compute-config -f workspace-compute-config.yaml

# 2. Create the workspace
anyscale workspace_v2 create \
  --name multi-model-pipeline-workspace \
  --image-uri anyscale/ray-llm:2.56.0-py312-cu130 \
  --compute-config workspace-compute-config:1 \
  --requirements requirements.txt

# 3. Start it (create does not auto-start) — or use the Anyscale console
anyscale workspace_v2 start -n multi-model-pipeline-workspace
```

See [`demo-guide.md`](demo-guide.md) → **Step 2: Set Up Your Workspace** for the full walkthrough.

### Local development

```bash
pip install -r requirements.txt
serve run serve_app:app

# In another terminal
python client.py
```

### Deploy to Anyscale

```bash
anyscale service deploy -f service.yaml
python client.py --url https://your-service.anyscale.com --token YOUR_ANYSCALE_TOKEN
```

### Throughput test (observe scaling)

```bash
python client.py --throughput 20
```

Watch autoscaling live from the Anyscale dashboard, or programmatically with the interactive [`observability-runbook.ipynb`](observability-runbook.ipynb) — it polls Anyscale's metrics and logs (service health, replica counts, request rates, and recent controller/replica logs).

## Deployments

| Deployment | Resource | Latency | Replicas | Scaling Behavior |
|---|---|---|---|---|
| **ContentFilter** | CPU only | ~1 ms | 1–8 | Handles high volume cheaply. First in pipeline — rejects bad content before GPU cost. |
| **SentimentClassifier** | 0.25 GPU + 1 CPU | ~10 ms | 1–4 | Up to 4 replicas share 1 A10 GPU. Scales on `num_ongoing_requests`. |
| **Summarizer** | 0.25 GPU + 1 CPU | ~200 ms | 1–4 | Up to 4 replicas share 1 A10 GPU. Scales on `num_ongoing_requests`. |
| **ContentPipeline** | CPU only | N/A | 1–5 | Orchestrator. Mostly waiting on downstream RPCs. Scales on connection count. |

## Models

Both models are **ungated** — no HuggingFace account, license acceptance, or token required. They download automatically on first run.

| Model | HuggingFace ID | Params | Size | Task |
|---|---|---|---|---|
| DistilBERT | [`distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) | 67 M | ~250 MB | Sentiment analysis |
| DistilBART-CNN | [`sshleifer/distilbart-cnn-12-6`](https://huggingface.co/sshleifer/distilbart-cnn-12-6) | 306 M | ~1.2 GB | Text summarization |

## Instance Types

Worker nodes are auto-selected by Anyscale (`auto_select_worker_config: true`). Both GPU models are lightweight (~1.5 GB combined) and share fractional GPU on A10 nodes.

| Role | Instance Type | Specs |
|---|---|---|
| Head node | `16CPU-64GB` | 16 vCPU, 64 GB RAM, CPU-only (`Standard_D16s_v5`) |
| GPU workers | `1xA10-24G:36CPU-440GB` | 36 vCPU, 440 GB RAM, 1× A10 (24 GB VRAM) |
| CPU workers | `16CPU-64GB` | 16 vCPU, 64 GB RAM, CPU-only (`Standard_D16s_v5`) |

## Design Reasoning

<details>
<summary>Why each function exists (click to expand)</summary>

The code in `serve_app.py` has detailed `WHY` comments inline. Summary:

| Function / Method | Reasoning |
|---|---|
| `ContentFilter.filter()` | Cheap CPU gate (~1 ms). Rejects bad content *before* reaching expensive GPU deployments. |
| `ContentFilter.__init__` regex compile | Regex compilation is expensive vs. matching. Compile once at startup, amortize across thousands of requests/sec. |
| `SentimentClassifier.classify()` | Wraps HF pipeline in `asyncio.to_thread` because HF `__call__` is synchronous and holds the GIL. |
| `SentimentClassifier.__init__` warmup | First inference triggers JIT + CUDA kernel caching. Warmup moves that latency out of the request path. |
| `Summarizer.summarize()` | Wraps HF summarization pipeline in `asyncio.to_thread`. DistilBART-CNN is purpose-built for summarization and ungated. |
| `Summarizer.__init__` warmup | Same as SentimentClassifier — moves first-inference latency out of the request path. |
| `ContentPipeline.analyze()` order | Filter first (1 ms CPU) → parallel GPU fan-out. Sequential would double latency; filter-last would waste GPU on rejected content. |
| `asyncio.gather()` in analyze | Sentiment and summary are independent RPCs. `gather()` dispatches both immediately; total latency = `max()` not `sum()`. |
| `.bind()` graph | Ray Serve dependency injection. Defines the deployment graph at import time; handles are injected as constructor args with automatic load balancing. |

</details>

## Customization

<details>
<summary>Swap the summarizer model</summary>

Change `_MODEL_ID` in the `Summarizer` class:

```python
class Summarizer:
    _MODEL_ID = "facebook/bart-large-cnn"  # Larger, higher quality (~1.6 GB)
```

</details>

<details>
<summary>Swap the sentiment model</summary>

Change `_MODEL_ID` in the `SentimentClassifier` class:

```python
class SentimentClassifier:
    _MODEL_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"  # 3-class
```

</details>

<details>
<summary>Add a new deployment</summary>

1. Define a new `@serve.deployment` class with its own scaling config
2. Add it as a parameter to `ContentPipeline.__init__`
3. Wire it up in the `app = ContentPipeline.bind(...)` call
4. Call it from the orchestrator via `self.new_deployment.method.remote(...)`

</details>

## Files

| File | Description |
|---|---|
| **[`demo.ipynb`](demo.ipynb)** | **Step-by-step notebook:** deploy, wait, call `/health` & `/analyze`, test pipeline |
| [`observability-runbook.ipynb`](observability-runbook.ipynb) | Interactive runbook — polls Anyscale metrics & logs to observe service health and autoscaling |
| [`serve_app.py`](serve_app.py) | Ray Serve application — 4 deployments with inline WHY reasoning |
| [`service.yaml`](service.yaml) | Anyscale service config (auto-select compute) |
| [`compute_config_azure.yaml`](compute_config_azure.yaml) | Compute config for `odl_user_2298227_cloud` (A10 GPU + CPU pools); registered as `multi-modal-2298227` |
| [`workspace-compute-config.yaml`](workspace-compute-config.yaml) | CPU-only compute config for the dev workspace; registered as `workspace-compute-config` |
| [`create_workspace.sh`](create_workspace.sh) | Helper script that creates the workspace and pushes this folder into it |
| [`client.py`](client.py) | Test client with functional and throughput tests |
| [`requirements.txt`](requirements.txt) | Python dependencies |
| [`demo-guide.md`](demo-guide.md) | Extended guide and troubleshooting for the demo |
| [`LICENSE`](LICENSE) | MIT license |

## Key Ray Serve Concepts Demonstrated

| Concept | What it does |
|---|---|
| `@serve.deployment` | Each class becomes an independently managed set of replicas |
| `DeploymentHandle` | Inter-deployment RPCs via `.remote()` calls |
| `autoscaling_config` | Per-deployment min/max replicas and scaling triggers |
| `ray_actor_options` | Per-deployment CPU/GPU resource allocation |
| `@serve.ingress` | FastAPI integration for the HTTP-facing deployment |
| `.bind()` | Wiring the deployment dependency graph at application definition time |
| `asyncio.gather` | Running independent downstream calls in parallel |

## References

- [Ray Serve — Model Composition](https://docs.ray.io/en/latest/serve/model_composition.html)
- [Ray Serve — Autoscaling Guide](https://docs.ray.io/en/latest/serve/autoscaling-guide.html)
- [Ray Serve — Resource Allocation](https://docs.ray.io/en/latest/serve/resource-allocation.html)
- [Anyscale Services](https://docs.anyscale.com/services/)
- [Anyscale on Azure](https://docs.anyscale.com/cloud-deployment/azure/)